# Оценка моделей без дообучения (zero-shot baselines)

Снимаем метрики ROUGE-1/2/L, BERTScore и LLM-as-a-judge для моделей из `offline_assets`,
**без fine-tuning**, чтобы сравнить с дообученными версиями.

Модели:
- `ai-forever/FRED-T5-large` (seq2seq, zero-shot)
- `Qwen/Qwen2.5-3B-Instruct` (causal, zero-shot)
- `Qwen/Qwen2.5-7B-Instruct` (causal, zero-shot)

In [1]:
import json
import os
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
from bert_score import score as bert_score_fn
from datasets import load_from_disk
from IPython.display import display
from rouge_score import rouge_scorer
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/gab1k/llm-hse/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: Tesla V100-PCIE-32GB


In [2]:
MODELS_CONFIG = {
    "fred-t5-large-zeroshot": {
        "path": "../offline_assets/ai-forever__FRED-T5-large",
        "model_type": "seq2seq",
        "params_M": 820,
    },
    "qwen2.5-3b-zeroshot": {
        "path": "../offline_assets/Qwen__Qwen2.5-3B-Instruct",
        "model_type": "causal_zeroshot",
        "params_M": 3000,
    },
    "qwen2.5-7b-zeroshot": {
        "path": "../offline_assets/Qwen__Qwen2.5-7B-Instruct",
        "model_type": "causal_zeroshot",
        "params_M": 7000,
    },
}

SEQ2SEQ_PREFIX = "Summarize: "
NUM_EVAL_SAMPLES = 24
BATCH_SIZE = 8

LENGTH_MODES_SEQ2SEQ = {
    "short":  {"max_new_tokens": 60,  "min_new_tokens": 20, "num_beams": 4},
    "medium": {"max_new_tokens": 100, "min_new_tokens": 50, "num_beams": 4},
    "long":   {"max_new_tokens": 150, "min_new_tokens": 80, "num_beams": 4},
}

LENGTH_MODES_CAUSAL = {
    "short":  {"max_new_tokens": 120, "temperature": 0.3, "do_sample": False},
    "medium": {"max_new_tokens": 220, "temperature": 0.3, "do_sample": False},
    "long":   {"max_new_tokens": 350, "temperature": 0.4, "do_sample": True},
}

CAUSAL_INSTRUCTIONS = {
    "short":  "Напиши КРАТКОЕ резюме статьи в 2-3 предложения.\n\n",
    "medium": "Напиши резюме статьи в 4-5 предложений.\n\n",
    "long":   "Напиши ПОДРОБНОЕ резюме статьи в 6-8 предложений.\n\n",
}

In [3]:
dataset = load_from_disk("../offline_assets/gazeta_dataset")
test_data = dataset["test"].select(range(NUM_EVAL_SAMPLES))
sources = list(test_data["text"])
references = list(test_data["summary"])
print(f"Тестовых примеров: {len(sources)}")

Тестовых примеров: 24


In [4]:
def tokenize_simple(text):
    return re.findall(r'\b\w+\b', text.lower())

def get_ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def compression_ratio(source, summary):
    sw = len(source.split())
    return len(summary.split()) / sw if sw > 0 else 0.0

def novelty_score(source, summary, n=1):
    src_ng = set(get_ngrams(tokenize_simple(source), n).keys())
    sum_ng = list(get_ngrams(tokenize_simple(summary), n).keys())
    if not sum_ng:
        return 0.0
    return sum(1 for ng in sum_ng if ng not in src_ng) / len(sum_ng)


class _RuTokenizer:
    """Токенизатор для rouge_score с поддержкой кириллицы."""
    def tokenize(self, text):
        return re.findall(r'\w+', text.lower())


def build_causal_prompt(text, mode):
    instruction = CAUSAL_INSTRUCTIONS[mode]
    if len(text) > 3500:
        text = text[:3500] + "..."
    return (
        f"<|im_start|>system\n"
        f"Ты — эксперт по созданию резюме новостных статей на русском языке. "
        f"Пиши чётко, информативно, без воды.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{instruction}"
        f"Статья:\n{text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def generate_seq2seq_batch(model, tokenizer, texts, mode, batch_size=BATCH_SIZE):
    params = LENGTH_MODES_SEQ2SEQ[mode]
    preds = []
    model.eval()
    for i in tqdm(range(0, len(texts), batch_size), desc=f"seq2seq [{mode}]", leave=False):
        batch = [SEQ2SEQ_PREFIX + t for t in texts[i:i+batch_size]]
        enc = tokenizer(batch, max_length=600, truncation=True, padding=True, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**enc, no_repeat_ngram_size=3, early_stopping=True, **params)
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds


def generate_causal_single(model, tokenizer, texts, mode):
    params = LENGTH_MODES_CAUSAL[mode]
    eos_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    preds = []
    model.eval()
    for text in tqdm(texts, desc=f"causal [{mode}]", leave=False):
        prompt = build_causal_prompt(text, mode)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1500).to(DEVICE)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs, no_repeat_ngram_size=3, repetition_penalty=1.1,
                eos_token_id=eos_id, pad_token_id=tokenizer.pad_token_id, **params,
            )
        summary = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
        preds.append(summary.replace("<|im_end|>", "").strip())
    return preds


def compute_all_metrics(src_list, pred_list, ref_list):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], tokenizer=_RuTokenizer())
    agg = {k: [] for k in ['rouge1', 'rouge2', 'rougeL']}
    for ref, pred in zip(ref_list, pred_list):
        scores = scorer.score(ref, pred)
        for k in agg:
            agg[k].append(scores[k].fmeasure)
    rouge_scores = {k: round(np.mean(v) * 100, 3) for k, v in agg.items()}

    comp = [compression_ratio(s, p) for s, p in zip(src_list, pred_list)]
    nov1 = [novelty_score(s, p, 1) for s, p in zip(src_list, pred_list)]
    nov2 = [novelty_score(s, p, 2) for s, p in zip(src_list, pred_list)]

    return {
        **rouge_scores,
        "compression_ratio": round(np.mean(comp), 4),
        "novelty_1gram_%": round(np.mean(nov1) * 100, 2),
        "novelty_2gram_%": round(np.mean(nov2) * 100, 2),
        "avg_pred_words": round(np.mean([len(p.split()) for p in pred_list]), 1),
    }

## Генерация и ROUGE / custom метрики

In [5]:
all_results = {}
all_predictions = {}

for model_name, cfg in MODELS_CONFIG.items():
    model_path = cfg["path"]
    model_type = cfg["model_type"]

    if not os.path.exists(model_path):
        print(f"Модель {model_name} не найдена: {model_path}, пропускаем")
        continue

    print(f"\n{'=' * 60}")
    print(f"{model_name} ({model_type}, {cfg['params_M']}M params)")

    try:
        if model_type == "seq2seq":
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_path, torch_dtype=torch.float16
            ).to(DEVICE)
        else:
            tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_path, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
            )
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

        all_results[model_name] = {}
        all_predictions[model_name] = {}

        for mode in ["short", "medium", "long"]:
            if model_type == "seq2seq":
                preds = generate_seq2seq_batch(model, tokenizer, sources, mode)
            else:
                preds = generate_causal_single(model, tokenizer, sources, mode)

            metrics = compute_all_metrics(sources, preds, references)
            all_results[model_name][mode] = metrics
            all_predictions[model_name][mode] = preds
            print(f"  [{mode}] ROUGE-1: {metrics['rouge1']:.2f}, "
                  f"ROUGE-2: {metrics['rouge2']:.2f}, "
                  f"ROUGE-L: {metrics['rougeL']:.2f}")

        del model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  Ошибка: {e}")
        import traceback; traceback.print_exc()


fred-t5-large-zeroshot (seq2seq, 820M params)


`torch_dtype` is deprecated! Use `dtype` instead!


  [short] ROUGE-1: 0.00, ROUGE-2: 0.00, ROUGE-L: 0.00


  [medium] ROUGE-1: 0.00, ROUGE-2: 0.00, ROUGE-L: 0.00


  [long] ROUGE-1: 0.00, ROUGE-2: 0.00, ROUGE-L: 0.00

qwen2.5-3b-zeroshot (causal_zeroshot, 3000M params)


...

  [medium] ROUGE-1: 12.03, ROUGE-2: 0.71, ROUGE-L: 8.61


  [long] ROUGE-1: 11.82, ROUGE-2: 0.79, ROUGE-L: 8.22

qwen2.5-7b-zeroshot (causal_zeroshot, 7000M params)


  [short] ROUGE-1: 14.44, ROUGE-2: 1.68, ROUGE-L: 10.50


  [medium] ROUGE-1: 15.62, ROUGE-2: 2.38, ROUGE-L: 11.88


  [long] ROUGE-1: 14.80, ROUGE-2: 1.75, ROUGE-L: 10.48


## Таблица ROUGE + custom метрики

In [6]:
rows = []
for mname, modes in all_results.items():
    for mode, m in modes.items():
        rows.append({
            "Модель": mname,
            "Режим": mode,
            "ROUGE-1": m.get("rouge1"),
            "ROUGE-2": m.get("rouge2"),
            "ROUGE-L": m.get("rougeL"),
            "Comp. Ratio": m.get("compression_ratio"),
            "Novelty-1 (%)": m.get("novelty_1gram_%"),
            "Novelty-2 (%)": m.get("novelty_2gram_%"),
            "Avg words": m.get("avg_pred_words"),
        })

df_rouge = pd.DataFrame(rows)
display(df_rouge)

,Модель,Режим,ROUGE-1,ROUGE-2,ROUGE-L,Comp. Ratio,Novelty-1 (%),Novelty-2 (%),Avg words
0,fred-t5-large-zeroshot,short,0.000,0.000,0.000,0.0920,99.31,100.00,48.9
1,fred-t5-large-zeroshot,medium,0.000,0.000,0.000,0.1565,99.44,100.00,83.1
2,fred-t5-large-zeroshot,long,0.000,0.000,0.000,0.2388,99.44,100.00,126.9
3,qwen2.5-3b-zeroshot,short,14.004,1.087,10.674,0.0788,67.85,96.77,42.2
4,qwen2.5-3b-zeroshot,medium,12.030,0.710,8.614,0.1168,74.86,97.47,64.7
5,qwen2.5-3b-zeroshot,long,11.824,0.787,8.218,0.1280,75.08,98.05,69.3
6,qwen2.5-7b-zeroshot,short,14.440,1.684,10.498,0.0657,68.01,96.50,35.2
7,qwen2.5-7b-zeroshot,medium,15.621,2.380,11.879,0.0923,66.88,95.60,51.3
8,qwen2.5-7b-zeroshot,long,14.803,1.754,10.483,0.1422,69.34,95.36,75.0


## BERTScore

In [7]:
bertscore_results = {}

for model_name in all_predictions:
    preds = all_predictions[model_name].get("medium", [])
    if not preds:
        continue
    print(f"BERTScore для {model_name}...")
    P, R, F1 = bert_score_fn(
        preds, references[:len(preds)],
        lang="ru", device=DEVICE, batch_size=32, verbose=False,
    )
    bertscore_results[model_name] = {
        "P": round(P.mean().item() * 100, 3),
        "R": round(R.mean().item() * 100, 3),
        "F1": round(F1.mean().item() * 100, 3),
    }
    print(f"  F1: {bertscore_results[model_name]['F1']:.2f}")

display(pd.DataFrame(bertscore_results).T)

BERTScore для fred-t5-large-zeroshot...
  F1: 49.72
BERTScore для qwen2.5-3b-zeroshot...
  F1: 66.00
BERTScore для qwen2.5-7b-zeroshot...
  F1: 67.74


,P,R,F1
fred-t5-large-zeroshot,48.140,51.439,49.722
qwen2.5-3b-zeroshot,65.028,67.053,66.001
qwen2.5-7b-zeroshot,66.894,68.671,67.737


## LLM-as-a-judge

Используем Qwen2.5-7B-Instruct в качестве LLM-судьи для оценки качества саммари.
Судья оценивает по 5-балльной шкале:
- **Фактологическая точность** (1-5)
- **Полнота** (1-5)
- **Связность** (1-5)

In [8]:
JUDGE_MODEL_PATH = "../offline_assets/Qwen__Qwen2.5-7B-Instruct"
NUM_JUDGE_SAMPLES = 30

JUDGE_PROMPT_TEMPLATE = """<|im_start|>system
Ты — эксперт по оценке качества текстовых резюме на русском языке.
Оцени качество предложенного резюме по трём критериям по шкале от 1 до 5.
Ответь ТОЛЬКО в формате JSON, без пояснений:
{{"accuracy": <1-5>, "completeness": <1-5>, "coherence": <1-5>}}<|im_end|>
<|im_start|>user
Исходная статья (начало):
{source}

Эталонное резюме:
{reference}

Оцениваемое резюме:
{prediction}<|im_end|>
<|im_start|>assistant
"""

print(f"Загружаем модель-судью: {JUDGE_MODEL_PATH}")
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_PATH, trust_remote_code=True)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_PATH, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token
judge_model.eval()
print("Модель-судья загружена")

Загружаем модель-судью: ../offline_assets/Qwen__Qwen2.5-7B-Instruct
Модель-судья загружена


In [9]:
import json as json_module

def run_llm_judge(source, reference, prediction):
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        source=source[:1500],
        reference=reference,
        prediction=prediction,
    )
    inputs = judge_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = judge_model.generate(
            **inputs, max_new_tokens=100, temperature=0.1, do_sample=False,
            eos_token_id=judge_tokenizer.convert_tokens_to_ids("<|im_end|>"),
            pad_token_id=judge_tokenizer.pad_token_id,
        )
    response = judge_tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    response = response.replace("<|im_end|>", "").strip()

    try:
        # Пытаемся извлечь JSON из ответа
        json_match = re.search(r'\{[^}]+\}', response)
        if json_match:
            scores = json_module.loads(json_match.group())
            return {
                "accuracy": int(scores.get("accuracy", 0)),
                "completeness": int(scores.get("completeness", 0)),
                "coherence": int(scores.get("coherence", 0)),
            }
    except (json_module.JSONDecodeError, ValueError):
        pass
    return None


judge_results = {}

for model_name in all_predictions:
    preds = all_predictions[model_name].get("medium", [])
    if not preds:
        continue

    print(f"\nLLM-judge для {model_name}...")
    scores_list = []

    for i in tqdm(range(min(NUM_JUDGE_SAMPLES, len(preds)))):
        result = run_llm_judge(sources[i], references[i], preds[i])
        if result:
            scores_list.append(result)

    if scores_list:
        judge_results[model_name] = {
            "accuracy": round(np.mean([s["accuracy"] for s in scores_list]), 2),
            "completeness": round(np.mean([s["completeness"] for s in scores_list]), 2),
            "coherence": round(np.mean([s["coherence"] for s in scores_list]), 2),
            "valid_responses": len(scores_list),
            "total_samples": NUM_JUDGE_SAMPLES,
        }
        avg = np.mean([judge_results[model_name]["accuracy"],
                       judge_results[model_name]["completeness"],
                       judge_results[model_name]["coherence"]])
        judge_results[model_name]["avg_score"] = round(avg, 2)
        print(f"  accuracy={judge_results[model_name]['accuracy']}, "
              f"completeness={judge_results[model_name]['completeness']}, "
              f"coherence={judge_results[model_name]['coherence']}, "
              f"avg={judge_results[model_name]['avg_score']}")
    else:
        print(f"  Не удалось получить валидные оценки")


LLM-judge для fred-t5-large-zeroshot...


100%|██████████| 24/24 [00:37<00:00,  1.55s/it]


  accuracy=1.79, completeness=1.08, coherence=1.08, avg=1.32

LLM-judge для qwen2.5-3b-zeroshot...


100%|██████████| 24/24 [00:36<00:00,  1.54s/it]


  accuracy=2.92, completeness=2.5, coherence=2.62, avg=2.68

LLM-judge для qwen2.5-7b-zeroshot...


100%|██████████| 24/24 [00:36<00:00,  1.51s/it]

  accuracy=3.08, completeness=2.71, coherence=2.83, avg=2.87


In [10]:
if judge_results:
    df_judge = pd.DataFrame(judge_results).T
    display(df_judge)
else:
    print("Нет результатов LLM-judge")

,accuracy,completeness,coherence,valid_responses,total_samples,avg_score
fred-t5-large-zeroshot,1.79,1.08,1.08,24.0,30.0,1.32
qwen2.5-3b-zeroshot,2.92,2.50,2.62,24.0,30.0,2.68
qwen2.5-7b-zeroshot,3.08,2.71,2.83,24.0,30.0,2.87


## Итоговая сводная таблица (zero-shot baselines)

In [12]:
os.makedirs("./results", exist_ok=True)

summary_rows = []
for model_name, modes in all_results.items():
    cfg = MODELS_CONFIG[model_name]
    med = modes.get("medium", {})
    bs = bertscore_results.get(model_name, {})
    jg = judge_results.get(model_name, {})

    summary_rows.append({
        "Модель": model_name,
        "Параметры (M)": cfg["params_M"],
        "ROUGE-1": med.get("rouge1", "-"),
        "ROUGE-2": med.get("rouge2", "-"),
        "ROUGE-L": med.get("rougeL", "-"),
        "BERTScore F1": bs.get("F1", "-"),
        "LLM-judge avg": jg.get("avg_score", "-"),
        "Novelty-1 (%)": med.get("novelty_1gram_%", "-"),
        "Comp. Ratio": med.get("compression_ratio", "-"),
    })

final_df = pd.DataFrame(summary_rows)
display(final_df)

,Модель,Параметры (M),ROUGE-1,ROUGE-2,ROUGE-L,BERTScore F1,LLM-judge avg,Novelty-1 (%),Comp. Ratio
0,fred-t5-large-zeroshot,820,0.000,0.00,0.000,49.722,1.32,99.44,0.1565
1,qwen2.5-3b-zeroshot,3000,12.030,0.71,8.614,66.001,2.68,74.86,0.1168
2,qwen2.5-7b-zeroshot,7000,15.621,2.38,11.879,67.737,2.87,66.88,0.0923


In [13]:
EXAMPLE_IDX = [0, 5, 20]

for idx in EXAMPLE_IDX:
    if idx >= len(sources):
        continue
    print(f"\n{'=' * 70}")
    print(f"ПРИМЕР #{idx}")
    print(f"ИСХОДНЫЙ ТЕКСТ (начало): {sources[idx][:300]}...")
    print(f"\nЭТАЛОН: {references[idx]}")
    for mname in all_predictions:
        preds = all_predictions[mname].get("medium", [])
        if idx < len(preds):
            print(f"\n{mname}: {preds[idx]}")


ПРИМЕР #0
ИСХОДНЫЙ ТЕКСТ (начало): На этих выходных в Берлине прошли крупные акции протеста против введенных для борьбы с коронавирусом ограничений. Демонстранты скандировали «Путин!» По словам депутата городской палаты представителей Гуннара Линдеманна («Альтернатива для Германии»), люди выкрикивали фамилию российского президента из...

ЭТАЛОН: Протестующие против антикоронавирусных мер немцы скандировали имя российского президента, потому что уважают его. Такое мнение выразил депутат городской палаты представителей Гуннар Линдеманн. На этих выходных в Берлине прошли крупные акции протеста. Манифестанты требовали отменить ношение масок и отказаться от соблюдения безопасного расстояния в 1,5 м друг от друга.

fred-t5-large-zeroshot:  менстру менстру менстру алхими алхими алхимихими алхими алхимиохимиохимиохими алхими алхими общежития общежития общежития потенциала потенциала потенциала потенциал потенциала потенциал потенциал потенциал потенциала потенциала эффективности эффективности

In [14]:
del judge_model
torch.cuda.empty_cache()
print("Память GPU освобождена")

Память GPU освобождена
